# Güncellenmiş Hücreler — v6 Uyumlu

`03_nihai_model.ipynb` içindeki dört hücre eski sürümlerden (V0/V1 dönemi) kalmıştı
ve v6 nihai modeliyle tutarlı değildi. Bu defter onların güncel sürümlerini içerir.

## Ne Değişti

### HÜCRE 6C — K-Fold Çapraz Doğrulama

Eski hücre, model seçimini `SONUC_JSON` üzerinden **V0 mimari karşılaştırmasından**
yapıyordu; yani v6 çalıştırıldığında bile `yolo11m-seg` seçip 8'lik yığınla fp16
eğitiyordu. Tezde raporlanan K-fold sonucu (0,6391 ± 0,0078, 3 kat) bu ayarlarla
yeniden üretilemez.

| Parametre | Eski (V0 kalıntısı) | Yeni (v6) | Neden |
|---|---|---|---|
| Model | otomatik → `yolo11m-seg` | `yolo11l-seg` | nihai model bu |
| `K` | 5 | **3** | tezde 3 kat raporlandı |
| `batch` | 8 | **6** | 960 px'te yolo11l-seg için bellek sınırı |
| `amp` | ayarlanmamış → `True` (fp16) | **`False`** | ⚠️ fp16 maske kaybında NaN taşması yapıyordu |
| `close_mosaic` | 15 | **20** | Hücre 5 (v6) ile aynı |
| `box` | 7.5 | **8.0** | Hücre 5 (v6) ile aynı |
| `cls` | 0.6 | **0.5** | Hücre 5 (v6) ile aynı |
| `patience` | 20 | **35** | Hücre 5 (v6) ile aynı |
| `ABLATION_ETIKET` | `'v4'` | `'v6_fp32'` | sonuç dosyası adı |

`amp` en kritik olanı: v6 eğitiminde 83. devirde fp16 taşması yaşandı. K-fold fp16
ile koşarsa aynı hataya düşebilir ve sonuçlar nihai modelle karşılaştırılamaz.

### v6 Gerçek Dünya Testi

Model yolu artık sabit ve açık (`hasar_yolo11l-seg_v6_fp32`); V3 ile karşılaştırma
değerleri koda gömülü değil, parametre olarak veriliyor.

### Tespit Edilemeyen Görseller

Önceki sürüm, kendisinden önceki hücrenin bıraktığı `model` ve `hic_bulamadi`
değişkenlerine bağlıydı; tek başına çalışmıyordu. Artık kendi modelini yüklüyor ve
değişken yoksa anlaşılır bir uyarı veriyor.

### HÜCRE 9 / 10 — Rapor ve Toplu İşlem

Rapor üreticisi **çift dilli** hale getirildi (`DIL = 'TR' | 'EN'`). Toplu işlem de
aynı anahtarı kullanıyor; Excel sayfa ve sütun başlıkları da dile göre değişiyor.

---
*Hücreleri `03_nihai_model.ipynb` içindeki karşılıklarının yerine yapıştırın.*

## 1. HÜCRE 6C — K-Fold (v6)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔁 HÜCRE 6C (v6) — K-FOLD ÇAPRAZ DOĞRULAMA
# ═══════════════════════════════════════════════════════════════════════════════
#
# 🎯 NE YAPAR?
#    Train+Valid havuzunu K parçaya böler. Her turda 1 parça doğrulama, K-1 parça
#    eğitim olur. Test kümesine DOKUNULMAZ. Sonuç: ortalama ± standart sapma.
#
# ⚠️ V0 KALINTISI DÜZELTİLDİ:
#    Eski sürüm modeli V0 mimari karşılaştırmasından seçiyordu (yolo11m-seg, batch 8,
#    fp16). Artık v6 reçetesi sabit: yolo11l-seg, batch 6, amp=False.
#
# ⏱️ K=3 × 60 devir ≈ 4-5 saat (A100). Süreyi kısmak için KFOLD_EPOCHS düşürün.
# ═══════════════════════════════════════════════════════════════════════════════

import os, shutil, yaml, json, time, random
import numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

# ══ v6 AYARLARI (Hücre 5 / v6-fp32 ile birebir aynı olmalı) ═══════════════════
K             = 3                 # tezde raporlanan kat sayısı
KFOLD_MODEL   = 'yolo11l-seg'     # nihai mimari — otomatik seçim KAPALI
KFOLD_BATCH   = 6                 # 960 px'te yolo11l-seg
KFOLD_EPOCHS  = 60                # tam eğitimden kısa: K-fold maliyet dengesi
KFOLD_AMP     = False             # ⚠️ fp32 — fp16 maske kaybında NaN taşıyor
ABLATION_ETIKET = globals().get('ABLATION_ETIKET', 'v6_fp32')

assert 'DATASET_DIR' in globals(), 'Önce Hücre 0B / 1 çalıştırılmalı.'
assert 'IMG_SIZE' in globals(),    'IMG_SIZE tanımsız — Hücre 1.'
assert 'HASAR_SINIFLAR' in globals(), 'HASAR_SINIFLAR tanımsız — Hücre 1.'

print('=' * 70)
print(f'🔁 {K}-FOLD ÇAPRAZ DOĞRULAMA  (v6)')
print('=' * 70)
print(f'   Model       : {KFOLD_MODEL}')
print(f'   Batch       : {KFOLD_BATCH}   |  Çözünürlük: {IMG_SIZE}')
print(f'   Devir/kat   : {KFOLD_EPOCHS}  |  amp: {KFOLD_AMP}')
print(f'   Etiket      : {ABLATION_ETIKET}')
print('=' * 70)

# ══ 1. HAVUZ ══════════════════════════════════════════════════════════════════
havuz = []
for split in ['train', 'valid']:
    idir = os.path.join(DATASET_DIR, split, 'images')
    ldir = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(idir):
        continue
    for ip in Path(idir).glob('*.*'):
        lp = os.path.join(ldir, ip.stem + '.txt')
        if os.path.exists(lp):
            havuz.append((str(ip), lp))
if not havuz:
    raise RuntimeError('❌ Havuz boş!')

random.seed(42); random.shuffle(havuz)
test_n = len(list(Path(os.path.join(DATASET_DIR, 'test', 'images')).glob('*.*')))
print(f'\n📦 Havuz: {len(havuz)} görsel (train+valid)')
print(f'   Test ({test_n} görsel) dokunulmadı.')

# ══ 2. BÖL ════════════════════════════════════════════════════════════════════
fb = len(havuz) // K
foldlar = [havuz[i*fb:(i+1)*fb] for i in range(K)]
if len(havuz) % K:
    foldlar[-1].extend(havuz[K*fb:])
print(f'📐 Kat boyutları: {[len(f) for f in foldlar]}')

# ══ 3. EĞİT ═══════════════════════════════════════════════════════════════════
KFOLD_DIR  = '/content/kfold_calisma'
KFOLD_JSON = os.path.join(ANA_DIZIN, f'kfold_{KFOLD_MODEL}_{ABLATION_ETIKET}.json')

kf_sonuc = {}
if os.path.exists(KFOLD_JSON):
    with open(KFOLD_JSON) as f:
        kf_sonuc = json.load(f)
    print(f'\n📂 Önceki kat sonuçları: {len(kf_sonuc)}/{K}')

gt0 = time.time()
for fn in range(K):
    if str(fn) in kf_sonuc:
        print(f'\n⏭️  Kat {fn+1} atlandı (zaten var)')
        continue

    print(f'\n{"─"*70}\n🔁 KAT {fn+1}/{K}\n{"─"*70}')
    shutil.rmtree(KFOLD_DIR, ignore_errors=True)
    for s in ['train', 'valid']:
        os.makedirs(f'{KFOLD_DIR}/{s}/images', exist_ok=True)
        os.makedirs(f'{KFOLD_DIR}/{s}/labels', exist_ok=True)

    for i, fold in enumerate(foldlar):
        hedef = 'valid' if i == fn else 'train'
        for ip, lp in fold:
            i_ = f'{KFOLD_DIR}/{hedef}/images/{os.path.basename(ip)}'
            l_ = f'{KFOLD_DIR}/{hedef}/labels/{os.path.basename(lp)}'
            try:
                if not os.path.exists(i_): os.symlink(ip, i_)
                if not os.path.exists(l_): os.symlink(lp, l_)
            except OSError:
                if not os.path.exists(i_): shutil.copy(ip, i_)
                if not os.path.exists(l_): shutil.copy(lp, l_)

    print(f'   Train: {len(os.listdir(f"{KFOLD_DIR}/train/images"))} | '
          f'Valid: {len(os.listdir(f"{KFOLD_DIR}/valid/images"))}')

    fyaml = f'{KFOLD_DIR}/data.yaml'
    with open(fyaml, 'w') as f:
        yaml.dump({'train': f'{KFOLD_DIR}/train/images',
                   'val':   f'{KFOLD_DIR}/valid/images',
                   'nc': len(HASAR_SINIFLAR), 'names': HASAR_SINIFLAR},
                  f, default_flow_style=False, sort_keys=False)

    t0 = time.time()
    mdl = YOLO(f'{KFOLD_MODEL}.pt')
    mdl.train(
        data=fyaml, epochs=KFOLD_EPOCHS, imgsz=IMG_SIZE, batch=KFOLD_BATCH,
        device=DEVICE, project=MODEL_DIZIN,
        name=f'kfold_{KFOLD_MODEL}_{ABLATION_ETIKET}_f{fn+1}', exist_ok=True,
        pretrained=True, optimizer='AdamW', lr0=0.001, lrf=0.01,
        cos_lr=True, warmup_epochs=3.0,
        patience=35, workers=4, cache=False,
        amp=KFOLD_AMP,                       # ⚠️ v6 ile aynı — fp32
        # ── HÜCRE 5 (v6) İLE BİREBİR AYNI ÇOĞALTMA ──
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
        fliplr=0.5, flipud=0.0, degrees=5.0,
        mosaic=0.5, close_mosaic=20, copy_paste=0.3,
        scale=0.4, translate=0.1, erasing=0.2,
        box=8.0, cls=0.5, dfl=1.5,
        plots=False, verbose=False,
    )

    best = YOLO(os.path.join(MODEL_DIZIN,
                f'kfold_{KFOLD_MODEL}_{ABLATION_ETIKET}_f{fn+1}', 'weights', 'best.pt'))
    mt = best.val(data=fyaml, split='val', imgsz=IMG_SIZE, verbose=False)

    sinif_map = {}
    for ci, sn in enumerate(HASAR_SINIFLAR):
        try:
            sinif_map[sn] = float(mt.seg.class_result(ci)[2])
        except Exception:
            pass

    kf_sonuc[str(fn)] = {
        'fold': fn + 1,
        'mask_mAP50':    float(mt.seg.map50),
        'mask_mAP50_95': float(mt.seg.map),
        'box_mAP50':     float(mt.box.map50),
        'kesinlik':      float(mt.seg.mp),
        'duyarlilik':    float(mt.seg.mr),
        'sinif_mAP50':   sinif_map,
        'sure_dk':       round((time.time() - t0) / 60, 1),
        'n_train': len(os.listdir(f'{KFOLD_DIR}/train/images')),
        'n_valid': len(os.listdir(f'{KFOLD_DIR}/valid/images')),
    }
    with open(KFOLD_JSON, 'w') as f:
        json.dump(kf_sonuc, f, indent=2)
    print(f'   ✅ Kat {fn+1}: mask mAP50 = {mt.seg.map50:.4f} '
          f'({kf_sonuc[str(fn)]["sure_dk"]} dk)')

# ══ 4. ÖZET ═══════════════════════════════════════════════════════════════════
df = pd.DataFrame([kf_sonuc[k] for k in sorted(kf_sonuc, key=int)])
m, s = df['mask_mAP50'].mean(), df['mask_mAP50'].std()
print('\n' + '=' * 70)
print(f'📊 {K}-FOLD SONUÇ  ({KFOLD_MODEL}, {ABLATION_ETIKET})')
print('=' * 70)
print(df[['fold', 'mask_mAP50', 'mask_mAP50_95', 'kesinlik', 'duyarlilik',
          'sure_dk']].to_string(index=False))
print(f'\n   Maske mAP50 : {m:.4f} ± {s:.4f}')
print(f'   Değişim kats: %{100*s/m:.2f}')
print(f'   Toplam süre  : {(time.time()-gt0)/60:.0f} dk')
print(f'   Kayıt        : {KFOLD_JSON}')

## 2. v6 Gerçek Dünya Testi

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔍 v6 GERÇEK DÜNYA TESTİ — Görülmemiş, Etiketsiz Veri
# ═══════════════════════════════════════════════════════════════════════════════
#
# ⚠️ GÜNCELLEME: Model yolu artık sabit ve açık. Karşılaştırma değerleri koda gömülü
#    değil — KARSILASTIRMA sözlüğünden geliyor, istediğiniz sürümle kıyaslayın.
#
# ETİKET YOK → mAP hesaplanamaz. Ölçülenler: güven dağılımı, görsel başına tespit,
# eşik simülasyonu.
# ═══════════════════════════════════════════════════════════════════════════════

import os, random, time
import numpy as np
from pathlib import Path
from collections import Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt

# ══ AYARLAR ═══════════════════════════════════════════════════════════════════
V6_PATH   = os.path.join(MODEL_DIZIN, 'hasar_yolo11l-seg_v6_fp32', 'weights', 'best.pt')
KAYNAK    = '/content/drive/MyDrive/train'     # etiketsiz havuz
ORNEK     = 300
ESIK      = 0.25
TTA       = False

# Kıyaslanacak önceki sürüm (tezdeki değerler) — gömülü değil, buradan değiştirin
KARSILASTIRMA = {'ad': 'V3 (yolo11m-seg, 960 px)', 'dusuk_guven_orani': 0.58}

assert os.path.exists(V6_PATH), f'Model yok: {V6_PATH}  (Hücre 5 / v6-fp32)'
model = YOLO(V6_PATH)
print(f'✅ Model: {os.path.basename(os.path.dirname(os.path.dirname(V6_PATH)))}')
print(f'   Sınıflar: {model.names}')

gorseller = [p for p in Path(KAYNAK).rglob('*')
             if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
if not gorseller:
    raise RuntimeError(f'❌ Görsel yok: {KAYNAK}')
random.seed(42)
sec = random.sample(gorseller, min(ORNEK, len(gorseller)))
print(f'📁 {len(gorseller)} görselden {len(sec)} örnek\n')

tum_conf, tespit_sayi, sinif_say, hic_bulamadi = [], [], Counter(), []
t0 = time.time()
for i, p in enumerate(sec, 1):
    r = model.predict(str(p), conf=ESIK, imgsz=IMG_SIZE, augment=TTA,
                      verbose=False, retina_masks=True)[0]
    n = 0 if r.boxes is None else len(r.boxes)
    tespit_sayi.append(n)
    if n == 0:
        hic_bulamadi.append(str(p))
    else:
        for j in range(n):
            tum_conf.append(float(r.boxes.conf[j]))
            sinif_say[r.names[int(r.boxes.cls[j])]] += 1
    if i % 50 == 0:
        print(f'   {i}/{len(sec)}  ({time.time()-t0:.0f} sn)')

tum_conf = np.array(tum_conf)
dusuk = float((tum_conf < 0.30).mean()) if len(tum_conf) else 0.0

print('\n' + '=' * 70)
print('📊 v6 GERÇEK DÜNYA ÖZETİ')
print('=' * 70)
print(f'   Görsel                : {len(sec)}')
print(f'   Tespit yapılan        : {len(sec)-len(hic_bulamadi)} '
      f'(%{100*(len(sec)-len(hic_bulamadi))/len(sec):.1f})')
print(f'   Hiç tespit yok        : {len(hic_bulamadi)}')
print(f'   Toplam tespit         : {len(tum_conf)}')
print(f'   Görsel başına ortalama: {np.mean(tespit_sayi):.2f}')
if len(tum_conf):
    print(f'   Güven ortalama/medyan : {tum_conf.mean():.3f} / {np.median(tum_conf):.3f}')
    print(f'   <0,30 güven oranı     : %{100*dusuk:.1f}   '
          f"({KARSILASTIRMA['ad']}: %{100*KARSILASTIRMA['dusuk_guven_orani']:.0f})")
print(f'   Sınıf dağılımı        : {dict(sinif_say)}')

if len(tum_conf):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    ax[0].hist(tum_conf, bins=30, color='#2EA8C7', edgecolor='white')
    ax[0].axvline(0.30, color='#D1495B', ls='--', label='0,30')
    ax[0].set_title('Güven dağılımı'); ax[0].set_xlabel('güven'); ax[0].legend()
    esikler = np.arange(0.20, 0.91, 0.05)
    ax[1].plot(esikler, [(tum_conf >= e).sum() for e in esikler], 'o-', color='#E8A33D')
    ax[1].set_title('Eşiğe göre kalan tespit'); ax[1].set_xlabel('eşik')
    ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

print(f'\n💾 hic_bulamadi: {len(hic_bulamadi)} yol — sonraki hücre bunları inceler.')

## 3. Tespit Edilemeyen Görseller

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 👁️ TESPİT EDİLEMEYEN GÖRSELLERİ İNCELE
# ═══════════════════════════════════════════════════════════════════════════════
# ⚠️ GÜNCELLEME: Artık tek başına çalışır. Önceki hücreden `model` ve `hic_bulamadi`
#    gelmezse anlaşılır uyarı verir, sessizce patlamaz.
# ═══════════════════════════════════════════════════════════════════════════════

import os, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

DUSUK_ESIK = 0.05        # çok düşük eşikle tekrar dene
IZGARA     = (3, 3)

if 'hic_bulamadi' not in globals() or not hic_bulamadi:
    raise RuntimeError(
        '❌ `hic_bulamadi` yok ya da boş.\n'
        '   Önce "v6 GERÇEK DÜNYA TESTİ" hücresini çalıştırın.\n'
        '   (Boşsa iyi haber: model her görselde en az bir tespit yapmış.)')

if 'model' not in globals():
    V6_PATH = os.path.join(MODEL_DIZIN, 'hasar_yolo11l-seg_v6_fp32', 'weights', 'best.pt')
    assert os.path.exists(V6_PATH), f'Model yok: {V6_PATH}'
    model = YOLO(V6_PATH)
    print(f'ℹ️  Model yeniden yüklendi: {V6_PATH}')

n = IZGARA[0] * IZGARA[1]
random.seed(7)
secilen = random.sample(hic_bulamadi, min(n, len(hic_bulamadi)))
print(f'🔍 {len(hic_bulamadi)} görselden {len(secilen)} tanesi, conf={DUSUK_ESIK} ile:\n')

fig, axes = plt.subplots(*IZGARA, figsize=(18, 14))
axes = np.atleast_1d(axes).flatten()
for a in axes:
    a.axis('off')

hala_bos = 0
for idx, ip in enumerate(secilen):
    img = np.array(Image.open(ip).convert('RGB'))
    r = model.predict(img, conf=DUSUK_ESIK, imgsz=IMG_SIZE,
                      verbose=False, retina_masks=True)[0]
    k = 0 if r.boxes is None else len(r.boxes)
    hala_bos += (k == 0)
    axes[idx].imshow(r.plot()[:, :, ::-1] if k else img)
    axes[idx].set_title(f'conf={DUSUK_ESIK} → {k} tespit', fontsize=11,
                        fontweight='bold', color='#D1495B' if k == 0 else '#1e40af')
    axes[idx].axis('off')

plt.suptitle(f'Normal eşikte tespit yapılamayan görseller (conf={DUSUK_ESIK} ile tekrar)',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\n📌 Gösterilen {len(secilen)} görselin {hala_bos} tanesinde '
      f'düşük eşikte bile tespit yok.')
print('👁️  YORUM:')
print('   • Görseller gerçekten hasarsızsa → model DOĞRU davranıyor ✅')
print('   • Hasar var ama bulunamıyorsa    → duyarlılık (recall) sorunu ❌')

## 4. HÜCRE 9 — Çift Dilli Rapor

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📄 HÜCRE 9 (v2) — ÇİFT DİLLİ RAPOR ÜRETİCİ
# ═══════════════════════════════════════════════════════════════════════════════
# DIL = 'TR' → TRAMER formatında Türkçe rapor   (tez, savunma)
# DIL = 'EN' → aynı rapor İngilizce             (uluslararası yayın)
#
# Model sınıf adları Türkçe kalır (Cizik/Gocuk/Kirik); yalnızca GÖSTERİM çevrilir.
# Böylece model ağırlıkları ve diğer defterlerle uyum bozulmaz.
#
# Tam kaynak: rapor_cift_dilli.py  (bu defterin yanında)
# ═══════════════════════════════════════════════════════════════════════════════

# Colab'da: dosyayı /content'e yükleyip import edin
import sys
sys.path.insert(0, '/content')
from rapor_cift_dilli import (hasar_raporu_olustur, font_hazirla,
                              SINIF_AD, parca_ad, METIN)

font_hazirla('/content/fonts')

DIL = 'EN'          # ← 'TR' | 'EN'

pdf = hasar_raporu_olustur(
    sonuc, img_np,
    dil=DIL,
    arac_bilgi={
        'plaka': '34 ABC 123', 'marka': 'Dacia', 'model': 'Duster',
        'yil': 2020, 'renk': 'Black' if DIL == 'EN' else 'Siyah',
        'sase': 'WVWZZZ3CZLE000000', 'police_no': 'POL-2026-004521',
    },
    sistem_bilgi={
        'parca_model': 'YOLOv11m-seg (Carparts-Seg)',
        'hasar_model': 'YOLOv11l-seg (v6, fp32)',
        'mask_mAP50':  0.7238,
        'inference_ms': 8.84,
    },
    hasar_renk=HASAR_RENK,
    rapor_dir=RAPOR_DIR,
)
print(f'✅ {pdf}')

try:
    from google.colab import files
    files.download(pdf)
except Exception:
    pass

## 5. HÜCRE 10 — Toplu İşlem (çift dilli)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📦 HÜCRE 10 (v2) — TOPLU İŞLEM + EXCEL ÖZET  (çift dilli)
# ═══════════════════════════════════════════════════════════════════════════════
# ⚠️ GÜNCELLEME: DIL anahtarı eklendi. Excel sayfa/sütun başlıkları ve rapor dili
#    birlikte değişir. Sınıf adları gösterimde çevrilir, veri bozulmaz.
# ═══════════════════════════════════════════════════════════════════════════════

import os, time, shutil
import pandas as pd, numpy as np
from pathlib import Path
from rapor_cift_dilli import hasar_raporu_olustur, SINIF_AD, parca_ad

DIL           = 'EN'            # ← 'TR' | 'EN'
KAYNAK_KLASOR = os.path.join(DATASET_DIR, 'test', 'images') if DATASET_DIR else '/content/fotograflar'
LIMIT         = 20              # None = hepsi
PDF_URET      = True
GORSEL_URET   = True

B = {'TR': dict(dosya='Dosya', parca='Parça', hasar='Hasar Tipi', guven='Güven (%)',
                ortusme='Örtüşme (%)', alan='Alan (%)', yok='Tespit yok',
                s1='Tüm Tespitler', s2='Parça-Hasar Matrisi', s3='Etiket Özeti'),
     'EN': dict(dosya='File', parca='Part', hasar='Damage Type', guven='Confidence (%)',
                ortusme='Overlap (%)', alan='Area (%)', yok='No detection',
                s1='All Detections', s2='Part-Damage Matrix', s3='Label Summary')}[DIL]

gorseller = sorted([p for p in Path(KAYNAK_KLASOR).glob('*.*')
                    if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
if LIMIT:
    gorseller = gorseller[:LIMIT]
if not gorseller:
    raise RuntimeError(f'❌ Görsel yok: {KAYNAK_KLASOR}')

CIKTI = f'/content/toplu_islem_{DIL}'
shutil.rmtree(CIKTI, ignore_errors=True)
os.makedirs(f'{CIKTI}/pdf', exist_ok=True)
os.makedirs(f'{CIKTI}/gorsel', exist_ok=True)

print(f'📦 TOPLU İŞLEM — {len(gorseller)} görsel  |  dil: {DIL}')
print(f'   PDF: {PDF_URET}  |  Görsel: {GORSEL_URET}\n')

satirlar, t0 = [], time.time()
for i, p in enumerate(gorseller, 1):
    try:
        sonuc, img_np = fuzyon_analiz(str(p))          # Hücre 7
    except Exception as e:
        print(f'   ⚠️  {p.name}: {e}')
        continue

    if not sonuc['birlesik']:
        satirlar.append({B['dosya']: p.name, B['parca']: B['yok'],
                         B['hasar']: '-', B['guven']: 0, B['ortusme']: 0, B['alan']: 0})
    else:
        for b in sonuc['birlesik']:
            satirlar.append({
                B['dosya']:   p.name,
                B['parca']:   parca_ad(b['parca'], DIL),
                B['hasar']:   SINIF_AD[DIL].get(b['hasar'], b['hasar']),
                B['guven']:   round(b['guven'] * 100, 1),
                B['ortusme']: round(b.get('ortusme', 0) * 100, 1),
                B['alan']:    round(b['alan_yuzde'], 2),
            })

    if GORSEL_URET:
        gorsellestir(sonuc, img_np, kaydet=f'{CIKTI}/gorsel/{p.stem}.jpg')   # Hücre 8
    if PDF_URET:
        hasar_raporu_olustur(sonuc, img_np, dil=DIL, rapor_dir=f'{CIKTI}/pdf',
                             dosya_adi=f'{CIKTI}/pdf/{p.stem}_{DIL}.pdf',
                             hasar_renk=HASAR_RENK,
                             sistem_bilgi={'hasar_model': 'YOLOv11l-seg (v6, fp32)',
                                           'mask_mAP50': 0.7238, 'inference_ms': 8.84})
    if i % 5 == 0:
        hiz = i / (time.time() - t0)
        print(f'   {i}/{len(gorseller)} | {hiz:.2f} görsel/sn | '
              f'~{(len(gorseller)-i)/hiz:.0f} sn kaldı')

df = pd.DataFrame(satirlar)
gercek = df[df[B['parca']] != B['yok']]

xlsx = f'{CIKTI}/ozet_{DIL}.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    df.to_excel(w, sheet_name=B['s1'], index=False)
    if len(gercek):
        pd.crosstab(gercek[B['parca']], gercek[B['hasar']]).to_excel(w, sheet_name=B['s2'])
        (gercek.groupby([B['parca'], B['hasar']]).size()
               .reset_index(name='n').sort_values('n', ascending=False)
               .to_excel(w, sheet_name=B['s3'], index=False))

zip_yolu = shutil.make_archive(CIKTI, 'zip', CIKTI)
print(f'\n✅ TAMAMLANDI')
print(f'   İşlenen : {len(gorseller)}')
print(f'   Tespit  : {len(gercek)}')
print(f'   Süre    : {(time.time()-t0)/60:.1f} dk')
print(f'   Excel   : {xlsx}')
print(f'   Arşiv   : {zip_yolu}')

try:
    from google.colab import files
    files.download(zip_yolu)
except Exception:
    pass